In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [2]:
# =========================
# 1. LOAD DATA
# =========================
data = pd.read_csv("data.csv")

# Drop user_id
data = data.drop(columns=["user_id"])

In [3]:
# =========================
# 2. SPLIT FEATURES & LABEL
# =========================
X = data.iloc[:, :8].values   # fitur
y = data.iloc[:, 8:].values  # label (4 materi)

# =========================
# 3. NORMALISASI
# =========================
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

# =========================
# 4. TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
# =========================
# 5. CUSTOM LOSS
# =========================
def custom_loss(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))



In [5]:
# =========================
# 6. BUILD MODEL
# =========================
inputs = tf.keras.Input(shape=(X.shape[1],))

x = tf.keras.layers.Dense(64, activation='relu')(inputs)
x = tf.keras.layers.Dense(32, activation='relu')(x)

outputs = tf.keras.layers.Dense(4, activation='sigmoid')(x)

model = tf.keras.Model(inputs, outputs)



In [6]:
# =========================
# 7. COMPILE
# =========================
model.compile(
    optimizer='adam',
    loss=custom_loss,
    metrics=['mae']
)

model.summary()



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,788 (10.89 KB)

 Trainable params: 2,788 (10.89 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# =========================
# 8. CALLBACK
# =========================
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# =========================
# 9. TRAINING
# =========================
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=16,
    callbacks=[early_stop]
)



Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - loss: 0.2294 - mae: 0.4768 - val_loss: 0.3098 - val_mae: 0.5538
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step - loss: 0.2224 - mae: 0.4692 - val_loss: 0.3123 - val_mae: 0.5560
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - loss: 0.2157 - mae: 0.4618 - val_loss: 0.3149 - val_mae: 0.5583
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - loss: 0.2090 - mae: 0.4544 - val_loss: 0.3179 - val_mae: 0.5609
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - loss: 0.2025 - mae: 0.4470 - val_loss: 0.3213 - val_mae: 0.5637
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - loss: 0.1963 - mae: 0.4396 - val_loss: 0.3247 - val_mae: 0.5666


In [8]:
# =========================
# 10. EVALUATION
# =========================
loss, mae = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test MAE: {mae}")

# # =========================
# # 11. SAVE MODEL
# # =========================
# model.save("model/pretest_model")

# print("Model berhasil disimpan!")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.3098 - mae: 0.5538
Test Loss: 0.30980172753334045
Test MAE: 0.553829550743103
